# Fraud Project Final - Báo cáo có thể truy vết

## Câu hỏi kết luận
Mô hình có giúp ưu tiên giao dịch để kiểm tra hay không? Đọc cùng AP, precision/recall tại ngưỡng và số fraud tìm được khi chỉ kiểm tra Top-p. Notebook này tổng hợp cùng một lần chạy; số liệu chuẩn nằm trong `evaluation_summary.json`. AP dùng cách tổng có trọng số của scikit-learn, không nội suy hình thang diện tích PR. Dữ liệu hai ngày và phép chia ngẫu nhiên có phân tầng chỉ đánh giá nội bộ; chưa chứng minh khả năng dự báo giao dịch tương lai.

Notebook tổng hợp này được thực thi sau 01, 02 và 03 bởi `scripts/run_full_project.py`. Nó chỉ đọc các artifact của cùng lần chạy cuối để tránh vô tình chọn lại model/threshold sau khi đã mở test.

In [ ]:
from pathlib import Path
import json
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'config' / 'a1_config.json').is_file():
            return candidate
    raise FileNotFoundError('Không tìm thấy thư mục gốc dự án.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
from IPython.display import Image, display
import pandas as pd

audit = pd.read_csv(PROJECT_ROOT / 'outputs/tables/data_audit.csv')
pivot = pd.read_csv(PROJECT_ROOT / 'outputs/tables/train_time_class_pivot.csv')
comparison = pd.read_csv(PROJECT_ROOT / 'outputs/tables/model_comparison.csv')
top_p = pd.read_csv(PROJECT_ROOT / 'outputs/tables/top_p_metrics.csv')
evaluation = json.loads((PROJECT_ROOT / 'outputs/tables/evaluation_summary.json').read_text(encoding='utf-8'))
display(audit)
display(pivot.style.format({'fraud_rate': '{:.4%}'}))
display(comparison)
display(top_p)

In [ ]:
test = evaluation['test']
primary = test['cost_threshold_metrics']
print(f"Mô hình chính: {evaluation['selected_family']}")
print(f"Test AP: {test['average_precision']:.6f} (95% bootstrap CI {test['ap_bootstrap_95_ci'][0]:.6f} - {test['ap_bootstrap_95_ci'][1]:.6f})")
print(f"ROC-AUC: {test['roc_auc']:.6f}")
print(f"Threshold chi phí: {primary['threshold']:.8f}; Precision={primary['precision']:.4f}; Recall={primary['recall']:.4f}; F1={primary['f1']:.4f}")

In [ ]:
for name in ['class_distribution.png', 'amount_by_class.png', 'time_by_class.png', 'selected_correlations.png', 'eda_seaborn_multivariate.png', 'validation_pr_curve.png', 'test_pr_curve.png', 'test_confusion_matrix.png', 'top_p_performance.png', 'seaborn_error_cases_multivariate.png', 'feature_importance.png']:
    path = PROJECT_ROOT / 'outputs/figures' / name
    assert path.is_file(), path
    display(Image(filename=str(path)))

EDA trong `01_data_eda.ipynb` có nhận xét 2–3 câu ngay sau từng hình, gồm biểu đồ Seaborn đa biến và bảng pivot tạo trực tiếp từ train. Các hình validation/test ở đây chỉ tổng hợp artifact sau khi quyết định đã khóa; không dùng để chọn lại mô hình.

**Giới hạn:** dữ liệu chỉ bao phủ khoảng hai ngày, V1-V28 đã ẩn danh, không có lịch sử khách hàng và giả định chi phí là học thuật. Đầu ra dùng để xếp hạng ưu tiên kiểm tra, không tự động khóa thẻ.

## Diễn giải kết quả của lần chạy
Các con số dưới đây được tính từ output vừa tạo.

In [ ]:
primary = evaluation['test']['cost_threshold_metrics']
print(f"Ngưỡng chính tạo {primary['tp']+primary['fp']} cảnh báo; trong đó {primary['tp']} fraud và {primary['fp']} cảnh báo nhầm.")
print(f"Còn bỏ sót {primary['fn']} fraud; chi phí giả định mỗi giao dịch là {primary['expected_cost_per_transaction']:.6f}.")
print('Cấu hình đã giữ nguyên trước khi đọc test; kết quả này dùng để thảo luận giới hạn và ngân sách kiểm tra.')